<a href="https://colab.research.google.com/github/ysy3046/ai-data-bootcamp/blob/main/03_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 채점표 — 무료 개방 O / X 와 FVA

설계는 `분석설계.md`. 원점 2025-01-01(수)부터 7일 간격 52개, 7일을 원점 전날 한 번에 예측, 확장 윈도우(2023-01-01~).

| 채점 축 | 원점 수 |
|---|---|
| **무료 O** — 7일 안에 무료 개방일(명절 연휴 · 마지막 수요일) | 14 |
| **무료 X** | 38 |
| 2단: A 명절 연휴 주 · B 마지막 수요일 주 / C 평일 공휴일 주 · D 특별일 직후 주 · E 평소 주 | 4 · 10 / 4 · 13 · 21 |

지표: MASE(모델 비교, 같은 주의 계절 나이브와 비교) · WAPE(1순위) · 비용(최종 결정) · 적중률 · 80% 포함률 · 평균 구간 길이.
학습형 모델(MLP)은 시드 0~4 다섯 번의 평균이고, 차이가 시드 범위보다 작으면 「차이 없음」.

In [2]:
%pip -q install "holidays==0.103" pyarrow

from google.colab import drive
from pathlib import Path
import zipfile
import os

drive.mount("/content/drive")

WORK = Path("/content/drive/MyDrive/heritage_colab")
PROJECT = Path("/content/heritage")
PROJECT.mkdir(exist_ok=True)

zip_files = sorted(WORK.glob("heritage_colab*.zip"))
assert zip_files, "드라이브 heritage_colab 폴더에 원본 ZIP이 없어!"

with zipfile.ZipFile(zip_files[-1]) as archive:
    archive.extractall(PROJECT)

os.chdir(PROJECT)

# 채점 코드가 찾는 위치에 드라이브 결과 폴더 연결
result_dir = WORK / "run_v2"
assert (result_dir / "mlp_raw.parquet").exists(), "MLP 결과가 없어!"

local_results = PROJECT / "outputs" / "run_v2"
local_results.parent.mkdir(parents=True, exist_ok=True)

if not local_results.exists():
    local_results.symlink_to(result_dir, target_is_directory=True)

print("코드·결과 연결 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
코드·결과 연결 완료!


In [3]:
import sys; sys.path.insert(0, 'src')
import pandas as pd, numpy as np
import hv, dl
from pathlib import Path
pd.set_option('display.width', 240); pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.3f}'.format)
RUN = Path('outputs/run_v2')

v, w, c, s = hv.load()
assert hv.check_anchors(v, w, c)['일치'].all()
cal = hv.add_free_rule(hv.make_calendar())
panel = hv.build_panel(v, cal, s)
qden = hv.mase_denominators(panel)
wt = hv.week_table()
wt['무료'] = np.where(wt['무료 개방 주'], '무료 O', '무료 X')
print(wt['무료'].value_counts().to_dict(), wt['주범주'].value_counts().sort_index().to_dict())

{'무료 X': 38, '무료 O': 14} {'A 명절 연휴 주': 4, 'B 마지막 수요일 주': 10, 'C 평일 공휴일 주(무료 없음)': 4, 'D 특별일 직후 주': 13, 'E 평소 주': 21}


## 0. 예측 모으기

In [4]:
m7 = hv.backtest(panel, hv.fc_mean7, name='mean7')
mult = hv.service_level_multiplier(panel, weeks=26, by='site')      # 2024 하반기 검증 원점에서 고른 설정
BASE = {
    '계절 나이브': hv.backtest(panel, hv.fc_snaive7, name='계절 나이브'),
    '나이브': hv.backtest(panel, hv.fc_naive, name='나이브'),
    '1 직전 7일 평균': m7.assign(model='1 직전 7일 평균'),
    '2 + 서비스 수준 보정': hv.apply_multiplier(m7, mult, by='site', name='2 + 서비스 수준 보정'),
}
FVA_LABEL = {
    'past': '3 MLP: 과거 관람객만',
    '+dow': '4 + 요일·정기 휴관',
    '+holiday_nf': '5 + 공휴일(명절 아님)',
    '+myeongjeol': '6 FVA 1차: + 명절 연휴',
    '+lastwed': '7 FVA 2차: + 마지막 수요일',
    '+rain_prev': '8 + 전날 큰비',
    '[완벽예보]+rain': '9 [완벽한 예보 가정] + 당일 비',
    '(선택)+neighbor': '(선택) + 이웃 궁 휴관',
}
raw = pd.read_parquet(RUN / 'mlp_raw.parquet')
MLP = {lab: {sd: hv.attach(panel, g.drop(columns=['variant', 'seed']), lab)
             for sd, g in raw[raw['variant'] == vk].groupby('seed')}
       for vk, lab in FVA_LABEL.items() if (raw['variant'] == vk).any()}
CH = {lab: pd.read_parquet(RUN / f'chronos_{tag}.parquet').assign(model=lab)
      for tag, lab in [('zs', 'Chronos-2 제로샷'), ('hol', 'Chronos-2 + 공휴일'), ('free', 'Chronos-2 + 공휴일·무료 개방')]}
print('MLP 변형:', list(MLP), '| 시드:', sorted(raw['seed'].unique()), '| 원점:', raw['origin'].nunique())

MLP 변형: ['3 MLP: 과거 관람객만', '4 + 요일·정기 휴관', '5 + 공휴일(명절 아님)', '6 FVA 1차: + 명절 연휴', '7 FVA 2차: + 마지막 수요일', '8 + 전날 큰비', '9 [완벽한 예보 가정] + 당일 비', '(선택) + 이웃 궁 휴관'] | 시드: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | 원점: 52


In [5]:
def tag(bt):
    return bt.merge(wt[['origin', '무료', '주범주', '교안 구분']], on='origin', how='left')

def score(bt, by):
    """by 범주별 지표. 비용은 주당(원점당) 백만 원으로도."""
    bt = tag(bt)
    r = hv.summarize(bt, by=by, qden=qden)
    n = bt.groupby(by)['origin'].nunique()
    r['주당 비용(백만)'] = r['비용'].to_numpy() / 1e6 / n.reindex(r.index.get_level_values(0)).to_numpy()
    return r

def score_seeds(runs, by):
    """MLP: 시드별로 채점해 평균, MASE·비용의 시드 범위."""
    per = pd.concat([score(bt, by).assign(seed=sd) for sd, bt in runs.items()]).reset_index()
    keys = [by, 'model']
    m = per.groupby(keys).mean(numeric_only=True).drop(columns='seed')
    m['MASE 시드 범위'] = per.groupby(keys)['MASE'].agg(lambda x: x.max() - x.min())
    return m, per

FINAL_MLP = '7 FVA 2차: + 마지막 수요일'   # 최종 후보(무료 정보까지 넣은 행)

## 1. 교안 양식 채점표 — 무료 O / X

교안 평가표(방식 × 공휴일 없는 주 / 있는 주)와 같은 모양으로, 축만 **무료 개방 O / X** 로 바꿨습니다.

In [6]:
rows = [score(BASE[k], '무료') for k in ['계절 나이브', '나이브', '1 직전 7일 평균', '2 + 서비스 수준 보정']]
mlp_m, _ = score_seeds(MLP[FINAL_MLP], '무료'); rows.append(mlp_m.assign(**{'MASE 시드 범위': mlp_m['MASE 시드 범위']}))
rows += [score(bt, '무료') for bt in CH.values()]
T1 = pd.concat(rows)
order = ['계절 나이브', '나이브', '1 직전 7일 평균', '2 + 서비스 수준 보정', FINAL_MLP] + list(CH)
def wide(T, cols):
    out = T[cols].unstack(0)
    return out.reindex(order)
print('■ MASE (같은 주의 계절 나이브와 비교)'); display(wide(T1, ['MASE']))
print('■ 80% 포함률 · 평균 구간 길이(명) · q0.9 초과 비율'); display(wide(T1, ['포함률(80%)', '평균 구간 길이', 'q0.9 초과']))
print('■ 적중률(목표 0.8) · WAPE · 주당 비용(백만 원)'); display(wide(T1, ['적중률', 'WAPE', '주당 비용(백만)']))
T1.to_csv(RUN / 'table1_free_ox.csv', encoding='utf-8-sig')

■ MASE (같은 주의 계절 나이브와 비교)


MASE      
무료                     무료 O  무료 X
model                            
계절 나이브                1.780 1.055
나이브                   1.707 1.027
1 직전 7일 평균            1.574 1.095
2 + 서비스 수준 보정         1.574 1.095
7 FVA 2차: + 마지막 수요일   1.449 0.715
Chronos-2 제로샷         1.429 0.642
Chronos-2 + 공휴일       1.348 0.639
Chronos-2 + 공휴일·무료 개방 1.294 0.649

■ 80% 포함률 · 평균 구간 길이(명) · q0.9 초과 비율


포함률(80%)        평균 구간 길이           q0.9 초과      
무료                        무료 O  무료 X      무료 O      무료 X    무료 O  무료 X
model                                                                 
계절 나이브                     NaN   NaN       NaN       NaN     NaN   NaN
나이브                        NaN   NaN       NaN       NaN     NaN   NaN
1 직전 7일 평균                 NaN   NaN       NaN       NaN     NaN   NaN
2 + 서비스 수준 보정              NaN   NaN       NaN       NaN     NaN   NaN
7 FVA 2차: + 마지막 수요일      0.743 0.740 2,857.045 1,763.398   0.131 0.128
Chronos-2 제로샷            0.693 0.812 2,241.948 1,921.030   0.213 0.056
Chronos-2 + 공휴일          0.705 0.774 2,593.524 1,761.256   0.176 0.066
Chronos-2 + 공휴일·무료 개방    0.734 0.772 2,450.929 1,692.928   0.155 0.069

■ 적중률(목표 0.8) · WAPE · 주당 비용(백만 원)


적중률        WAPE       주당 비용(백만)       
무료                     무료 O  무료 X  무료 O  무료 X      무료 O   무료 X
model                                                         
계절 나이브                0.401 0.539 0.565 0.461    62.647 20.592
나이브                   0.296 0.439 0.528 0.371    58.857 22.487
1 직전 7일 평균            0.451 0.631 0.488 0.419    56.340 18.340
2 + 서비스 수준 보정         0.696 0.845 0.488 0.419    44.613 18.298
7 FVA 2차: + 마지막 수요일   0.791 0.781 0.405 0.301    32.543 13.484
Chronos-2 제로샷         0.683 0.851 0.439 0.249    42.269 10.854
Chronos-2 + 공휴일       0.730 0.837 0.400 0.251    34.822 10.953
Chronos-2 + 공휴일·무료 개방 0.745 0.827 0.362 0.253    30.519 11.022

## 2. 2단 범주 — A~E (주제 답변 · 진단)

In [7]:
rows = [score(BASE[k], '주범주') for k in ['계절 나이브', '1 직전 7일 평균', '2 + 서비스 수준 보정']]
m2, _ = score_seeds(MLP[FINAL_MLP], '주범주'); rows.append(m2)
rows += [score(bt, '주범주') for bt in CH.values()]
T2 = pd.concat(rows)
order2 = ['계절 나이브', '1 직전 7일 평균', '2 + 서비스 수준 보정', FINAL_MLP] + list(CH)
for col in ['MASE', '적중률', '주당 비용(백만)']:
    print('■', col); display(T2[col].unstack(0).reindex(order2))
T2.to_csv(RUN / 'table2_week_cats.csv', encoding='utf-8-sig')

■ MASE


주범주,A 명절 연휴 주,B 마지막 수요일 주,C 평일 공휴일 주(무료 없음),D 특별일 직후 주,E 평소 주
model,,,,,
계절 나이브,3.203,1.211,0.983,1.421,0.842
1 직전 7일 평균,2.564,1.179,1.049,1.390,0.921
2 + 서비스 수준 보정,2.564,1.179,1.049,1.390,0.921
7 FVA 2차: + 마지막 수요일,2.502,1.027,0.802,0.791,0.652
Chronos-2 제로샷,2.372,1.052,0.793,0.707,0.573
Chronos-2 + 공휴일,2.201,1.006,0.747,0.708,0.576
Chronos-2 + 공휴일·무료 개방,2.056,0.989,0.779,0.713,0.584


■ 적중률


주범주,A 명절 연휴 주,B 마지막 수요일 주,C 평일 공휴일 주(무료 없음),D 특별일 직후 주,E 평소 주
model,,,,,
계절 나이브,0.393,0.404,0.574,0.610,0.489
1 직전 7일 평균,0.409,0.468,0.566,0.744,0.575
2 + 서비스 수준 보정,0.589,0.739,0.835,0.899,0.813
7 FVA 2차: + 마지막 수요일,0.790,0.791,0.829,0.815,0.750
Chronos-2 제로샷,0.598,0.717,0.771,0.863,0.859
Chronos-2 + 공휴일,0.749,0.722,0.853,0.841,0.832
Chronos-2 + 공휴일·무료 개방,0.779,0.731,0.820,0.834,0.823


■ 주당 비용(백만)


주범주,A 명절 연휴 주,B 마지막 수요일 주,C 평일 공휴일 주(무료 없음),D 특별일 직후 주,E 평소 주
model,,,,,
계절 나이브,121.186,39.231,14.138,26.081,18.423
1 직전 7일 평균,102.661,37.811,15.176,19.849,18.008
2 + 서비스 수준 보정,89.374,26.709,12.407,24.265,15.726
7 FVA 2차: + 마지막 수요일,52.824,24.431,11.789,17.016,11.621
Chronos-2 제로샷,80.500,26.977,11.863,13.904,8.775
Chronos-2 + 공휴일,59.414,24.986,10.395,14.110,9.104
Chronos-2 + 공휴일·무료 개방,48.871,23.178,10.921,14.310,9.005


## 3. 교안 호환 — 공휴일 있는 주(11) / 없는 주(41)

In [8]:
rows = [score(BASE[k], '교안 구분') for k in ['계절 나이브', '나이브', '1 직전 7일 평균']]
m3, _ = score_seeds(MLP[FINAL_MLP], '교안 구분'); rows.append(m3)
rows += [score(bt, '교안 구분') for bt in CH.values()]
T3 = pd.concat(rows)
display(T3[['MASE', '포함률(80%)', '평균 구간 길이']].unstack(0).reindex(['계절 나이브', '나이브', '1 직전 7일 평균', FINAL_MLP] + list(CH)))

MASE          포함률(80%)           평균 구간 길이          
교안 구분                 공휴일 없는 주 공휴일 있는 주 공휴일 없는 주 공휴일 있는 주  공휴일 없는 주  공휴일 있는 주
model                                                                        
계절 나이브                   1.068    1.929      NaN      NaN       NaN       NaN
나이브                      1.020    1.921      NaN      NaN       NaN       NaN
1 직전 7일 평균               1.089    1.725      NaN      NaN       NaN       NaN
7 FVA 2차: + 마지막 수요일      0.738    1.564    0.751    0.704 1,809.822 2,980.680
Chronos-2 제로샷            0.671    1.535    0.809    0.672 1,975.706 2,128.742
Chronos-2 + 공휴일          0.668    1.433    0.767    0.712 1,790.567 2,709.523
Chronos-2 + 공휴일·무료 개방    0.671    1.386    0.771    0.728 1,739.318 2,484.864

## 4. FVA 표 — 전체 · 무료 O · 무료 X

행마다 앞 행에 정보를 하나 더했습니다. **6행(FVA 1차 명절 연휴)·7행(FVA 2차 마지막 수요일)의 비용 감소가 무료 O 열에 몰리면** 모델이 무료 정보를 제대로 쓴 것입니다. MLP 행은 시드 5개 평균, `5시드 ↓` 는 같은 시드끼리 앞 행과 비교해 다섯 번 모두 비용이 줄었는지입니다.

In [9]:
def cost_cols(bt):
    d = hv.add_cost(tag(bt)[lambda x: x['status'] == 'open'])
    return pd.Series({'비용 전체': d['cost'].sum(), '비용 무료 O': d.loc[d['무료'] == '무료 O', 'cost'].sum(),
                      '비용 무료 X': d.loc[d['무료'] == '무료 X', 'cost'].sum()})

fva_rows, seed_costs = {}, {}
for k in ['1 직전 7일 평균', '2 + 서비스 수준 보정']:
    r = hv.summarize(BASE[k], qden=qden).iloc[0]
    fva_rows[k] = pd.concat([r[['WAPE', 'MASE']], cost_cols(BASE[k])])
for lab, runs in MLP.items():
    per = pd.DataFrame({sd: pd.concat([hv.summarize(bt, qden=qden).iloc[0][['WAPE', 'MASE']], cost_cols(bt)])
                        for sd, bt in runs.items()}).T
    seed_costs[lab] = per
    row = per.mean(); row['MASE 시드 범위'] = per['MASE'].max() - per['MASE'].min()
    fva_rows[lab] = row
FVA = pd.DataFrame(fva_rows).T
chain = [k for k in FVA.index if not k.startswith('(선택)')]
for col in ['비용 전체', '비용 무료 O', '비용 무료 X']:
    FVA[f'앞 행 대비 {col[3:]}'] = FVA.loc[chain, col].diff()
# (선택) 행은 7행과 비교
if '(선택) + 이웃 궁 휴관' in FVA.index:
    for col in ['비용 전체', '비용 무료 O', '비용 무료 X']:
        FVA.loc['(선택) + 이웃 궁 휴관', f'앞 행 대비 {col[3:]}'] = FVA.loc['(선택) + 이웃 궁 휴관', col] - FVA.loc[FINAL_MLP, col]
labs = [l for l in chain if l in seed_costs]
prev = {labs[i]: labs[i - 1] for i in range(1, len(labs))}
prev['(선택) + 이웃 궁 휴관'] = FINAL_MLP
for col in ['비용 전체', '비용 무료 O', '비용 무료 X']:
    FVA[f'5시드 ↓ {col[3:]}'] = pd.Series({l: bool((seed_costs[l][col] < seed_costs[p][col]).all())
                                          for l, p in prev.items() if l in seed_costs})
won = [c for c in FVA.columns if c.startswith('비용') or c.startswith('앞 행')]
FVA_show = FVA.copy(); FVA_show[won] = (FVA_show[won] / 1e6).round(1)
FVA_show.to_csv(RUN / 'fva_v2.csv', encoding='utf-8-sig')
print('비용 단위: 백만 원'); FVA_show

비용 단위: 백만 원


,MASE,MASE 시드 범위,WAPE,비용 무료 O,비용 무료 X,비용 전체,앞 행 대비 전체,앞 행 대비 무료 O,앞 행 대비 무료 X,5시드 ↓ 전체,5시드 ↓ 무료 O,5시드 ↓ 무료 X
1 직전 7일 평균,1.224,NaN,0.444,788.800,696.900,"1,485.700",NaN,NaN,NaN,NaN,NaN,NaN
2 + 서비스 수준 보정,1.224,NaN,0.444,624.600,695.300,"1,319.900",-165.800,-164.200,-1.600,NaN,NaN,NaN
3 MLP: 과거 관람객만,0.966,0.025,0.377,669.000,547.600,"1,216.600",-103.300,44.400,-147.700,NaN,NaN,NaN
4 + 요일·정기 휴관,0.948,0.028,0.375,667.600,538.800,"1,206.400",-10.200,-1.400,-8.800,False,False,True
5 + 공휴일(명절 아님),0.941,0.017,0.367,650.000,516.500,"1,166.500",-39.900,-17.500,-22.300,True,True,True
6 FVA 1차: + 명절 연휴,0.915,0.026,0.341,484.700,500.400,985.100,-181.500,-165.300,-16.100,True,True,False
7 FVA 2차: + 마지막 수요일,0.913,0.028,0.339,455.600,512.400,968.000,-17.100,-29.100,12.100,False,False,False
8 + 전날 큰비,0.917,0.034,0.340,461.200,528.400,989.600,21.600,5.600,16.000,False,False,False
9 [완벽한 예보 가정] + 당일 비,0.907,0.034,0.336,463.700,528.000,991.700,2.100,2.500,-0.400,False,False,False
(선택) + 이웃 궁 휴관,0.917,0.028,0.338,457.700,524.700,982.400,14.400,2.100,12.300,False,False,False


In [10]:
# 참고: Chronos-2 세 행 (FVA 체인 밖) — + 공휴일 → + 공휴일·무료 개방 의 차이가 Chronos 쪽 무료 정보 가치
ch = pd.DataFrame({k: pd.concat([hv.summarize(bt, qden=qden).iloc[0][['WAPE', 'MASE']], cost_cols(bt) / 1e6]) for k, bt in CH.items()}).T
ch

,WAPE,MASE,비용 전체,비용 무료 O,비용 무료 X
Chronos-2 제로샷,0.318,0.854,"1,004.237",591.768,412.469
Chronos-2 + 공휴일,0.305,0.830,903.715,487.515,416.200
Chronos-2 + 공휴일·무료 개방,0.293,0.822,846.094,427.264,418.830


## 5. Chronos-2 공개일(2025-10-17) 전 / 후

채점 7일이 공개일 전에 끝나는 원점은 사전학습 자료와 겹쳤을 수 있습니다(교안). 공개일 후 원점만으로 따로 봅니다.

In [11]:
def rel(bt):
    return bt.assign(공개일=np.where(bt['origin'] + pd.Timedelta(days=6) < dl.CHRONOS_RELEASE, '공개일 전', '공개일 후'))
rows = [hv.summarize(rel(bt), by='공개일', qden=qden) for bt in CH.values()]
rows += [hv.summarize(rel(BASE['계절 나이브']), by='공개일', qden=qden)]
mm = pd.concat([hv.summarize(rel(bt), by='공개일', qden=qden).assign(seed=sd) for sd, bt in MLP[FINAL_MLP].items()])
rows.append(mm.reset_index().groupby(['공개일', 'model']).mean(numeric_only=True).drop(columns='seed'))
print('원점 수:', rel(BASE['나이브']).groupby('공개일')['origin'].nunique().to_dict())
pd.concat(rows)[['MASE', 'WAPE', '적중률', '비용']].unstack(0)

원점 수: {'공개일 전': 41, '공개일 후': 11}


MASE        WAPE         적중률                      비용                
공개일                   공개일 전 공개일 후 공개일 전 공개일 후 공개일 전 공개일 후             공개일 전           공개일 후
model                                                                                      
7 FVA 2차: + 마지막 수요일   0.914 0.908 0.345 0.321 0.781 0.791   701,482,990.062 266,524,873.164
Chronos-2 + 공휴일       0.833 0.818 0.316 0.276 0.807 0.812   678,408,661.311 225,305,986.958
Chronos-2 + 공휴일·무료 개방 0.822 0.823 0.299 0.276 0.803 0.810   622,700,722.597 223,393,017.461
Chronos-2 제로샷         0.856 0.846 0.329 0.289 0.801 0.819   763,746,233.521 240,490,790.583
계절 나이브                1.180 1.514 0.478 0.557 0.482 0.571 1,246,561,425.000 412,973,775.000

## 6. 원점별 — 원점이 적은 범주(A·C)와 계절 나이브보다 MASE 가 낮은 개소 비율

교안 규칙: 평균으로 정할 수 없을 때 원점마다 「계절 나이브보다 MASE 가 낮은 개소 비율」을 함께 적습니다.

In [12]:
def site_mae(bt):
    x = bt[bt['status'] == 'open'].assign(ae=lambda d: (d['y'] - d['yhat']).abs())
    return x.groupby(['origin', 'site'])['ae'].mean()
sn = site_mae(BASE['계절 나이브'])
cand = {'1 직전 7일 평균': BASE['1 직전 7일 평균'], FINAL_MLP + ' (시드 0)': MLP[FINAL_MLP][0], **CH}
beat = pd.DataFrame({k: (site_mae(bt) < sn.reindex(site_mae(bt).index)).groupby('origin').mean() for k, bt in cand.items()})
beat = beat.join(wt.set_index('origin')[['무료', '주범주']])
display(beat[beat['주범주'].str[0].isin(['A', 'C'])].round(2))
beat.groupby('무료').mean(numeric_only=True).round(2)

,1 직전 7일 평균,7 FVA 2차: + 마지막 수요일 (시드 0),Chronos-2 제로샷,Chronos-2 + 공휴일,Chronos-2 + 공휴일·무료 개방,무료,주범주
origin,,,,,,,
2025-01-01,0.170,0.570,0.740,0.870,0.870,무료 X,C 평일 공휴일 주(무료 없음)
2025-01-22,0.610,0.300,0.740,0.780,0.780,무료 O,A 명절 연휴 주
2025-01-29,0.780,0.700,0.960,1.000,1.000,무료 O,A 명절 연휴 주
2025-06-04,0.740,0.830,0.780,0.830,0.830,무료 X,C 평일 공휴일 주(무료 없음)
2025-08-13,0.040,0.170,0.090,0.300,0.170,무료 X,C 평일 공휴일 주(무료 없음)
2025-10-01,0.650,0.520,0.870,0.430,0.520,무료 O,A 명절 연휴 주
2025-10-08,0.870,0.960,0.910,0.910,0.910,무료 O,A 명절 연휴 주
2025-12-24,0.520,0.650,0.780,0.700,0.650,무료 X,C 평일 공휴일 주(무료 없음)


,1 직전 7일 평균,7 FVA 2차: + 마지막 수요일 (시드 0),Chronos-2 제로샷,Chronos-2 + 공휴일,Chronos-2 + 공휴일·무료 개방
무료,,,,,
무료 O,0.590,0.740,0.800,0.810,0.840
무료 X,0.420,0.710,0.800,0.800,0.800


## 7. 상반기 / 하반기 원점 (상반기는 학습 1.8년)

In [13]:
rows = [hv.summarize(bt.assign(반기=hv.half(bt)), by='반기', qden=qden) for bt in [BASE['계절 나이브'], BASE['1 직전 7일 평균'], *CH.values()]]
mm = pd.concat([hv.summarize(bt.assign(반기=hv.half(bt)), by='반기', qden=qden).assign(seed=sd) for sd, bt in MLP[FINAL_MLP].items()])
rows.append(mm.reset_index().groupby(['반기', 'model']).mean(numeric_only=True).drop(columns='seed'))
pd.concat(rows)[['MASE', 'WAPE', '적중률', '비용']].unstack(0)

MASE          WAPE           적중률                     비용                
반기                    상반기 원점 하반기 원점 상반기 원점 하반기 원점 상반기 원점 하반기 원점          상반기 원점          하반기 원점
model                                                                                          
1 직전 7일 평균             1.136  1.312  0.410  0.474  0.572  0.592 651,591,500.357 834,078,373.393
7 FVA 2차: + 마지막 수요일    0.899  0.927  0.337  0.341  0.778  0.789 418,619,217.149 549,388,646.077
Chronos-2 + 공휴일        0.812  0.848  0.298  0.312  0.802  0.814 384,093,606.329 519,621,041.941
Chronos-2 + 공휴일·무료 개방  0.802  0.843  0.284  0.301  0.801  0.808 356,002,565.296 490,091,174.762
Chronos-2 제로샷          0.831  0.877  0.303  0.332  0.799  0.811 416,464,502.509 587,772,521.594
계절 나이브                 1.183  1.317  0.458  0.535  0.488  0.514 718,212,675.000 941,322,525.000

In [14]:
from pathlib import Path
from google.colab import files
import shutil

result_dir = Path("/content/drive/MyDrive/heritage_colab/run_v2")

for name in [
    "table1_free_ox.csv",
    "table2_week_cats.csv",
    "fva_v2.csv",
]:
    assert (result_dir / name).exists(), f"{name}이 없어!"
    print(f"{name}: 저장 확인")

zip_path = shutil.make_archive(
    "/content/run_v2_scored_results",
    "zip",
    root_dir=str(result_dir),
)
files.download(zip_path)

table1_free_ox.csv: 저장 확인
table2_week_cats.csv: 저장 확인
fva_v2.csv: 저장 확인


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>